# Инференс

1. В Kaggle включите **GPU T4** и **Internet**.
2. Подключите dataset с `test.jsonl`: каждая строка должна содержать строковое поле `question`.
3. Нажмите **Run All**. Если найдено несколько `test.jsonl`, укажите нужный путь в первой ячейке.
4. Скачайте `/kaggle/working/predictions.jsonl` из Outputs.

Ноутбук загружает код инференса из закреплённого commit GitHub и `model.pt` из Google Drive,
проверяет SHA-256 весов и запускает обычный FP32 beam5. Токенизатор уже находится внутри checkpoint.
Основной замер — `generation_seconds`: генерация на готовой модели с синхронизацией CUDA.
Скачивание, установка, загрузка модели и токенизация в него не входят. Время подготовки и `total_seconds`
выводятся отдельно. Для замера в новом процессе перезапустите kernel.


In [ ]:
from pathlib import Path

REPOSITORY = "denakifev/nlp-codegen-transformer"
COMMIT = "f5509bf243ba297823ca99a9f17bbd1c3d77da36"
WEIGHTS_URL = "https://drive.google.com/uc?id=15iRZHsE0jEH8gOw7YywYC4xnL_Md719c"
WEIGHTS_SHA256 = "8c2c93e313f02a58155602febbcee5ce61c7b2f778f68715770af351e4899145"
TEST_PATH = None
DEVICE = "cuda:0"
BATCH_SIZE = 32
WORK_DIR = Path("/kaggle/working") if Path("/kaggle/working").is_dir() else Path.cwd() / "outputs/inference"


In [ ]:
import hashlib
import importlib.util
import json
import subprocess
import sys
import time
from urllib.request import urlopen

setup_started = time.perf_counter()
WORK_DIR.mkdir(parents=True, exist_ok=True)
CODE_DIR = WORK_DIR / "_runtime" / COMMIT / "src"
CHECKPOINT_PATH = WORK_DIR / "model.pt"
OUTPUT_PATH = WORK_DIR / "predictions.jsonl"
RUNTIME_SHA256 = {
    "__init__.py": "e3b0c44298fc1c149afbf4c8996fb92427ae41e4649b934ca495991b7852b855",
    "model.py": "f56e805528deb637646531980503c4b02a845d931d3f48fa0b5082892bf3350d",
    "decoding.py": "37108816e218fb7d4eec25c48771763bb1081e35f0f1dfea4fd7c5f3f8bc4398",
    "generation.py": "37deb0e5488923d1521614417b3010ba8ed3935c24324f867640272ea7177d6c",
    "inference.py": "649229472742f1723b867c8db626ed0888d6b3485c7437021410e980d876d13c"
}

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b""):
            digest.update(chunk)
    return digest.hexdigest()

for filename, expected in RUNTIME_SHA256.items():
    path = CODE_DIR / "nlp_codegen" / filename
    if path.is_file() and sha256(path) == expected:
        continue
    url = f"https://raw.githubusercontent.com/{REPOSITORY}/{COMMIT}/src/nlp_codegen/{filename}"
    with urlopen(url, timeout=30) as response:
        content = response.read()
    if hashlib.sha256(content).hexdigest() != expected:
        raise RuntimeError(f"Code checksum mismatch: {filename}")
    compile(content, filename, "exec")
    path.parent.mkdir(parents=True, exist_ok=True)
    temporary = path.with_suffix(".download")
    temporary.write_bytes(content)
    temporary.replace(path)
sys.path.insert(0, str(CODE_DIR))

DEPENDENCIES_READY = True
try:
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "--quiet", "gdown==5.2.0", "tokenizers>=0.21,<0.23"],
        check=True, capture_output=True, text=True, timeout=180,
    )
except (subprocess.CalledProcessError, subprocess.TimeoutExpired):
    DEPENDENCIES_READY = all(importlib.util.find_spec(name) for name in ("gdown", "tokenizers"))
    print("Dependency installation failed; checking available packages.")

WEIGHTS_READY = False
WEIGHTS_ERROR = None
try:
    if not DEPENDENCIES_READY:
        raise RuntimeError("gdown or tokenizers is unavailable")
    if not CHECKPOINT_PATH.is_file() or sha256(CHECKPOINT_PATH) != WEIGHTS_SHA256:
        temporary = CHECKPOINT_PATH.with_suffix(".download")
        print("Downloading model.pt from Google Drive...", flush=True)
        try:
            subprocess.run(
                [sys.executable, "-m", "gdown", WEIGHTS_URL, "--no-cookies", "--quiet", "-O", str(temporary)],
                check=True, capture_output=True, text=True, timeout=300,
            )
            if sha256(temporary) != WEIGHTS_SHA256:
                raise RuntimeError("Weights checksum mismatch")
            temporary.replace(CHECKPOINT_PATH)
        finally:
            temporary.unlink(missing_ok=True)
    WEIGHTS_READY = True
except (OSError, RuntimeError, subprocess.CalledProcessError, subprocess.TimeoutExpired) as error:
    WEIGHTS_ERROR = type(error).__name__
    print(f"Trained weights unavailable ({WEIGHTS_ERROR}). The last cell can run a random-weight timing check.")

if TEST_PATH is None:
    input_root = Path("/kaggle/input")
    candidates = sorted({path.resolve() for path in input_root.rglob("test.jsonl")}) if input_root.is_dir() else []
    if not candidates:
        candidates = sorted({path.resolve() for path in (Path.cwd() / "test.jsonl", WORK_DIR / "test.jsonl") if path.is_file()})
    TEST_PATH = candidates[0] if len(candidates) == 1 else None
    if TEST_PATH is None:
        print(f"Found {len(candidates)} test.jsonl files. Set TEST_PATH explicitly for predictions.")
else:
    TEST_PATH = Path(TEST_PATH)
print({"weights_ready": WEIGHTS_READY, "test_path": str(TEST_PATH) if TEST_PATH else None,
       "setup_seconds": round(time.perf_counter() - setup_started, 2)})


In [ ]:
if WEIGHTS_READY:
    if TEST_PATH is None or not TEST_PATH.is_file():
        raise FileNotFoundError("Attach a dataset with test.jsonl or set TEST_PATH in the first cell.")
    started_at = time.perf_counter()
    from nlp_codegen.inference import run_inference

    metrics = run_inference(
        CHECKPOINT_PATH, TEST_PATH, OUTPUT_PATH,
        device=DEVICE, precision="fp32", batch_size=BATCH_SIZE,
        num_beams=5, length_penalty=1.0, max_new_tokens=512,
        started_at=started_at,
    )
    print(json.dumps({name: metrics[name] for name in
                     ("output_path", "output_rows", "generation_seconds", "total_seconds", "gpu")}, indent=2))
else:
    print("Prediction inference skipped. This run does not write predictions.jsonl.")


## Если обученные веса не скачались

Следующая ячейка автоматически выполняется только при недоступности весов.
Она создаёт ту же архитектуру со случайной инициализацией и измеряет генерацию
на синтетическом батче: 32 последовательности по 128 входных токенов, до 64 новых токенов, beam5, FP32.
Это техническая проверка кода и GPU, **не оценка качества и не время полного test-инференса**.
Сохраняется только `random_benchmark.metrics.json`; случайные ответы в `predictions.jsonl` не записываются.


In [ ]:
if not WEIGHTS_READY:
    benchmark_started = time.perf_counter()
    import torch
    from nlp_codegen.model import CodeTransformer

    device = torch.device(DEVICE)
    if device.type != "cuda" or not torch.cuda.is_available():
        raise RuntimeError("Enable a T4 GPU to run the timing check.")
    torch.manual_seed(42)
    torch.backends.cuda.matmul.allow_tf32 = False
    torch.backends.cudnn.allow_tf32 = False
    torch.set_float32_matmul_precision("highest")
    model_config = {
        "d_model": 512, "d_ff": 1408, "encoder_layers": 6, "decoder_layers": 6,
        "num_heads": 8, "num_kv_heads": 8, "dropout": 0.05,
        "norm_type": "layernorm", "norm_eps": 1e-5, "rope_theta": 10000.0,
        "vocab_size": 16384, "pad_id": 0, "bos_id": 1, "eos_id": 2,
    }
    random_model = CodeTransformer(model_config).to(device=device, dtype=torch.float32).eval()
    sources = torch.randint(3, model_config["vocab_size"], (32, 128), device=device)
    sources[:, -1] = model_config["eos_id"]
    torch.cuda.synchronize(device)
    generation_started = time.perf_counter()
    with torch.inference_mode():
        generated = random_model.generate(sources, max_new_tokens=64, decoding="beam",
                                          num_beams=5, length_penalty=1.0)
    torch.cuda.synchronize(device)
    benchmark_metrics = {
        "weights": "random", "purpose": "synthetic_timing_only", "quality_measured": False,
        "gpu": torch.cuda.get_device_name(device), "precision": "fp32", "num_beams": 5,
        "batch_size": 32, "source_tokens": 128, "max_new_tokens": 64,
        "output_steps": generated.shape[1],
        "parameters": sum(parameter.numel() for parameter in random_model.parameters()),
        "generation_seconds": time.perf_counter() - generation_started,
        "total_seconds": time.perf_counter() - benchmark_started,
        "predictions_written": False,
    }
    (WORK_DIR / "random_benchmark.metrics.json").write_text(json.dumps(benchmark_metrics, indent=2) + "\n")
    print(json.dumps(benchmark_metrics, indent=2))
    del random_model, sources, generated
    torch.cuda.empty_cache()
else:
    print("Trained-weight inference completed; random-weight check is not needed.")
